In [1]:
from google.colab import drive
drive.mount('/content/drive')

from scipy.interpolate import interp1d
import pandas as pd
import numpy as np

train = pd.read_csv(
    "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"
)

EDGE_COLS = [
    "NIFTY27JAN2623800PE",
    "NIFTY27JAN2623900PE",
    "NIFTY27JAN2626400CE",
    "NIFTY27JAN2626500CE"
]

option_cols = train.columns[2:]

filled = train.copy()

for idx, row in train.iterrows():

    vals = row[option_cols].values.astype(float)

    x = np.arange(len(vals))

    valid = ~np.isnan(vals)

    if valid.sum() < 2:
        continue

    linear = interp1d(
        x[valid],
        vals[valid],
        kind="linear",
        fill_value="extrapolate"
    )

    for j, col in enumerate(option_cols):

        if np.isnan(vals[j]):

            pred = float(linear(j))

            if col in EDGE_COLS:

                series = train[col]

                prev_vals = series.iloc[max(0, idx-5):idx].dropna()

                next_vals = series.iloc[idx+1:idx+6].dropna()

                neigh = pd.concat(
                    [prev_vals, next_vals]
                )

                if len(neigh) > 0:

                    pred = neigh.mean()

            filled.loc[idx, col] = pred

filled.to_csv(
    "filled_dataset_edge_time.csv",
    index=False
)

print("DONE")

Mounted at /content/drive
DONE


In [2]:
import pandas as pd

ORIGINAL_DATASET_PATH = "/content/drive/MyDrive/IV_Surface/data/raw/dataset.csv"

SEPARATOR = "||"

def generate_solution(filled_path, output_path):

    original = pd.read_csv(ORIGINAL_DATASET_PATH)

    filled = pd.read_csv(filled_path)

    feature_cols = [
        c for c in original.columns
        if c != "datetime"
    ]

    rows = []

    for col in feature_cols:

        was_missing = original[col].isna()

        for idx in original.index[was_missing]:

            dt = original.loc[idx, "datetime"]

            uid = f"{dt}{SEPARATOR}{col}"

            val = filled.loc[idx, col]

            rows.append({
                "id": uid,
                "value": val
            })

    solution = pd.DataFrame(rows)

    solution = solution.sort_values(
        "id"
    ).reset_index(drop=True)

    solution.to_csv(
        "submission_edge_time.csv",
        index=False
    )

    print(solution.shape)
    print("SUBMISSION CREATED")

generate_solution(
    "filled_dataset_edge_time.csv",
    "submission_edge_time.csv"
)

(5460, 2)
SUBMISSION CREATED


In [3]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'filled_dataset_edge_time.csv', 'submission_edge_time.csv', 'drive', 'sample_data']


In [4]:
from google.colab import files

files.download("submission_edge_time.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>